In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore

In [ ]:
store=InMemoryStore()
user_id="u1"
user_details = ("user", user_id)

In [ ]:
store.put(
    user_details,
    "profile_1",
    {"data": "Name: Taqadus"}
)

store.put(
    user_details,
    "preference_1",
    {"data": "Prefer concise answers"}
)

store.put(
    user_details,
    "interest_1",
    {"data": "Interested in Artificial Intelligence and Machine Learning"}
)

store.put(
    user_details,
    "project_1",
    {"data": "Working on an AI-powered RAG application"}
)

store.put(
    user_details,
    "language_1",
    {"data": "Prefers English"}
)

store.put(
    user_details,
    "skill_1",
    {"data": "Python, LangChain, LangGraph and RAG"}
)

store.put(
    user_details,
    "goal_1",
    {"data": "Wants to become an AI/ML Engineer"}
)

store.put(
    user_details,
    "learning_1",
    {"data": "Currently learning LangGraph memory and agent workflows"}
)

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a helpful AI assistant.

Use the following information about the user when relevant:

{user_info}

Instructions:
- Personalize your responses when appropriate.
- Respect the user's preferences.
- Do not make up information about the user.
- Keep responses concise and helpful.
"""

In [ ]:
def chat_node(
    state: MessagesState,
    config: RunnableConfig,
    store: BaseStore
):
    user_id = config["configurable"]["user_id"]

    user_details = ("user", user_id)

    items = store.search(user_details)

    if items:
        user_details_content = "\n".join(
            f"- {item.value.get('data', '')}"
            for item in items
        )
    else:
        user_details_content = "No user information available."

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        user_info=user_details_content
    )

    messages = [
        SystemMessage(content=system_prompt)
    ] + state["messages"]

    response = model.invoke(messages)

    return {"messages": [response]}

In [ ]:
builder = StateGraph(MessagesState)

builder.add_node("chat", chat_node)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)

graph = builder.compile()

graph

In [ ]:
config = {
    "configurable": {
        "user_id": "u1"
    }
}

result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hi, what do you know about me?"
            }
        ]
    },
    config
)

print(result["messages"][-1].content)